# Comprehensive Data Analysis: Pediatric Bone Age Assessment

This notebook provides detailed exploration of X-ray hand images for automated bone age prediction using deep learning.

## Objectives
- Analyze image dataset characteristics
- Examine age and gender distributions  
- Assess data quality and preprocessing needs
- Prepare data pipeline for CNN training


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('default')
sns.set_style('whitegrid')

## Dataset Overview

In [ ]:
# Load metadata
import os
from PIL import Image

data_path = Path('../data')
csv_file = data_path / 'boneage-training-dataset.csv'

if csv_file.exists():
    df = pd.read_csv(csv_file)
    print(f"Dataset loaded: {len(df)} samples")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nFirst few rows:")
    print(df.head())
    
    # Create image paths
    if 'id' in df.columns:
        image_dir = data_path / 'images'
        df['image_path'] = df['id'].apply(lambda x: str(image_dir / f'{x}.png'))
        df['image_exists'] = df['image_path'].apply(lambda x: os.path.exists(x))
        print(f"\nImages found: {df['image_exists'].sum()}/{len(df)}")
else:
    print("CSV file not found. Please update the path.")
    df = None

## Data Statistics

In [ ]:
# Statistical analysis
if df is not None and 'boneage' in df.columns:
    print("=" * 60)
    print("BONE AGE STATISTICS")
    print("=" * 60)
    print(f"\nAge Range: {df['boneage'].min():.1f} - {df['boneage'].max():.1f} months")
    print(f"Mean Age: {df['boneage'].mean():.1f} months ({df['boneage'].mean()/12:.1f} years)")
    print(f"Median Age: {df['boneage'].median():.1f} months")
    print(f"Standard Deviation: {df['boneage'].std():.1f} months")
    
    # Gender distribution
    if 'male' in df.columns:
        gender_counts = df['male'].value_counts()
        print(f"\nGender Distribution:")
        print(f"  Male: {gender_counts.get(True, 0)} ({gender_counts.get(True, 0)/len(df)*100:.1f}%)")
        print(f"  Female: {gender_counts.get(False, 0)} ({gender_counts.get(False, 0)/len(df)*100:.1f}%)")
    
    # Age by gender
    if 'male' in df.columns:
        print(f"\nAge Statistics by Gender:")
        print(df.groupby('male')['boneage'].describe().round(1))

## Visualizations

In [ ]:
# Create comprehensive visualizations
if df is not None and 'boneage' in df.columns:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Age distribution histogram
    axes[0, 0].hist(df['boneage'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    axes[0, 0].set_title('Bone Age Distribution', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('Age (months)', fontsize=12)
    axes[0, 0].set_ylabel('Frequency', fontsize=12)
    axes[0, 0].axvline(df['boneage'].mean(), color='red', linestyle='--', 
                       label=f'Mean: {df["boneage"].mean():.1f} months')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Age by gender
    if 'male' in df.columns:
        male_ages = df[df['male'] == True]['boneage']
        female_ages = df[df['male'] == False]['boneage']
        axes[0, 1].hist([male_ages, female_ages], bins=30, label=['Male', 'Female'], 
                       alpha=0.7, edgecolor='black')
        axes[0, 1].set_title('Age Distribution by Gender', fontsize=14, fontweight='bold')
        axes[0, 1].set_xlabel('Age (months)', fontsize=12)
        axes[0, 1].set_ylabel('Frequency', fontsize=12)
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Box plot by gender
    if 'male' in df.columns:
        gender_labels = df['male'].map({True: 'Male', False: 'Female'})
        df_plot = df.copy()
        df_plot['Gender'] = gender_labels
        sns.boxplot(data=df_plot, x='Gender', y='boneage', ax=axes[1, 0], palette='Set2')
        axes[1, 0].set_title('Age Distribution by Gender (Box Plot)', fontsize=14, fontweight='bold')
        axes[1, 0].set_ylabel('Age (months)', fontsize=12)
        axes[1, 0].grid(True, alpha=0.3, axis='y')
    
    # 4. Sample images (if available)
    axes[1, 1].text(0.5, 0.5, 'Sample X-ray Images\n(Load images to display)', 
                    ha='center', va='center', fontsize=14, transform=axes[1, 1].transAxes)
    axes[1, 1].set_title('Sample Images', fontsize=14, fontweight='bold')
    axes[1, 1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Visualizations complete")